# 03 - Autoencoder models (A-D)

* **A** dense AE on SBERT, **B** LSTM seq2seq AE on GloVe, **C** denoising AE and VAE - all trained on **genuine
  reviews only**; fake score = reconstruction error.
* **D** hybrid: latent z + reconstruction error + style features → MLP / XGBoost (proposed).

This notebook shows *how the threshold is chosen* and the generated result tables/plots.

In [ ]:
# Make `src` importable and run from the repo root (works locally and on Colab)
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from IPython.display import Image, Markdown, display
from src.config import load_config
cfg = load_config()
print("repo root:", ROOT)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from src.features import load_bundle
from src.models import load_runner
ae, meta = load_runner("checkpoints/ae_dense__salminen__seed42")
data = load_bundle("salminen", meta["config"])
val = data.idx("val")
s_val = ae.score(data, val)
print("genuine train reviews used:", len(data.genuine_idx("train")), "| fakes seen in training: 0")

### Threshold selection on the validation split (then frozen for test)

In [ ]:
p, r, t = precision_recall_curve(data.y[val], s_val)
f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(t, f1[:-1], label="F1"); ax.plot(t, p[:-1], label="precision"); ax.plot(t, r[:-1], label="recall")
ax.axvline(meta["threshold"], color="k", ls="--", label=f"chosen threshold {meta['threshold']:.3f}")
ax.set(xlabel="reconstruction-error threshold", title="Model A: choosing the threshold on VALIDATION data")
ax.legend(); fig.tight_layout(); fig.savefig("results/plots/ae_threshold_selection.png", dpi=300); plt.show()

In [ ]:
for t in ["autoencoders", "ablation", "cross_domain"]:
    display(Markdown(open(f"results/tables/{t}.md", encoding="utf-8").read()))

In [ ]:
import glob
for f in sorted(glob.glob("results/plots/recon_error_hist_*.png")) + sorted(glob.glob("results/plots/latent_umap_*.png")):
    display(Markdown(f"**{f}**")); display(Image(f, width=700))

In [ ]:
display(Image("report/figures/hybrid_architecture.png", width=1000))
for f in ["shap_group_importance.png", "shap_beeswarm.png"]:
    display(Image(f"results/plots/explain/hybrid__salminen__seed42/{f}", width=750))